# CLAP 10s Chunk Embedding

Split every clip in `data/audio` into non-overlapping ≤10s chunks, embed each chunk with LAION-CLAP, and save vectors aligned with episode id, start/end timings, and episode-level YAMNet top-5 classes.

In [6]:
from pathlib import Path

import laion_clap
import librosa
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

AUDIO_DIR = Path("../data/audio")
YAMNET_CSV = Path("../data/yamnet_top5_classifications.csv")
OUT_EMB = Path("../data/clap_10s_embeddings.npy")
OUT_META = Path("../data/clap_10s_metadata.csv")

CHUNK_SEC = 10
SR = 48_000

YAMNET_COLS = [
    "yamnet_rank_1_class",
    "yamnet_rank_1_mean_score",
    "yamnet_rank_1_max_score",
    "yamnet_rank_2_class",
    "yamnet_rank_2_mean_score",
    "yamnet_rank_2_max_score",
    "yamnet_rank_3_class",
    "yamnet_rank_3_mean_score",
    "yamnet_rank_3_max_score",
    "yamnet_rank_4_class",
    "yamnet_rank_4_mean_score",
    "yamnet_rank_4_max_score",
    "yamnet_rank_5_class",
    "yamnet_rank_5_mean_score",
    "yamnet_rank_5_max_score",
    "yamnet_top5_json",
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE}")
print(f"AUDIO_DIR={AUDIO_DIR.resolve()} exists={AUDIO_DIR.is_dir()}")
print(f"YAMNET_CSV={YAMNET_CSV.resolve()} exists={YAMNET_CSV.is_file()}")

device=cuda
AUDIO_DIR=C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data\audio exists=True
YAMNET_CSV=C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data\yamnet_top5_classifications.csv exists=True


In [7]:
yamnet = pd.read_csv(YAMNET_CSV)
yamnet = yamnet.set_index("id")
missing = [c for c in YAMNET_COLS if c not in yamnet.columns]
if missing:
    raise ValueError(f"YAMNet CSV missing columns: {missing}")
yamnet = yamnet[YAMNET_COLS]
print(f"YAMNet rows: {len(yamnet):,}")
yamnet.head(2)

YAMNet rows: 6,374


,yamnet_rank_1_class,yamnet_rank_1_mean_score,yamnet_rank_1_max_score,yamnet_rank_2_class,yamnet_rank_2_mean_score,yamnet_rank_2_max_score,yamnet_rank_3_class,yamnet_rank_3_mean_score,yamnet_rank_3_max_score,yamnet_rank_4_class,yamnet_rank_4_mean_score,yamnet_rank_4_max_score,yamnet_rank_5_class,yamnet_rank_5_mean_score,yamnet_rank_5_max_score,yamnet_top5_json
id,,,,,,,,,,,,,,,,
1dff287a-bf34-4e80-9505-52766d2bfda8,Speech,0.751908,0.993948,Vehicle,0.037192,0.579699,Motor vehicle (road),0.019876,0.266377,Car,0.019566,0.252760,Music,0.018977,0.860701,"[{""rank"": 1, ""class_index"": 0, ""class_name"": ""..."
d30d00f8-633d-4c85-9f10-e41fc0ea4780,Speech,0.392640,0.968733,Vehicle,0.050055,0.394888,Motor vehicle (road),0.031567,0.199376,Child singing,0.028110,0.445738,Car,0.024970,0.184834,"[{""rank"": 1, ""class_index"": 0, ""class_name"": ""..."


In [8]:
audio_paths = sorted(AUDIO_DIR.glob("*.mp3"))
print(f"clips found: {len(audio_paths):,}")
if not audio_paths:
    raise FileNotFoundError(f"No .mp3 files under {AUDIO_DIR.resolve()}")
audio_paths[:5]

clips found: 6,374


[WindowsPath('../data/audio/00057e4c-6542-4bed-be04-a308002dda40.mp3'),
 WindowsPath('../data/audio/0012d176-c941-4c93-96cb-a710b0790c55.mp3'),
 WindowsPath('../data/audio/001b0e37-055a-4bda-b02d-02fa2b83812d.mp3'),
 WindowsPath('../data/audio/0021b5eb-c771-458e-9abf-c931b5b088d8.mp3'),
 WindowsPath('../data/audio/0029513c-e141-42c4-9a17-6bec518cacca.mp3')]

In [9]:
# PyTorch 2.6+ defaults torch.load(weights_only=True); LAION-CLAP ckpts need False.
_orig_torch_load = torch.load


def _torch_load_compat(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)


torch.load = _torch_load_compat
try:
    clap_model = laion_clap.CLAP_Module(enable_fusion=False, device=DEVICE)
    clap_model.load_ckpt()
    clap_model.eval()
finally:
    torch.load = _orig_torch_load

print("CLAP loaded")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2755.24it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Load our best checkpoint in the paper.
The checkpoint is already downloaded
Load Checkpoint...
logit_scale_a 	 Loaded
logit_scale_t 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_real.weight 	 Loaded
audio_branch.spectrogram_extractor.stft.conv_imag.weight 	 Loaded
audio_branch.logmel_extractor.melW 	 Loaded
audio_branch.bn0.weight 	 Loaded
audio_branch.bn0.bias 	 Loaded
audio_branch.patch_embed.proj.weight 	 Loaded
audio_branch.patch_embed.proj.bias 	 Loaded
audio_branch.patch_embed.norm.weight 	 Loaded
audio_branch.patch_embed.norm.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm1.weight 	 Loaded
audio_branch.layers.0.blocks.0.norm1.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.relative_position_bias_table 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.qkv.bias 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.weight 	 Loaded
audio_branch.layers.0.blocks.0.attn.proj.bias 	 Loaded
audio_branch.layers.0.blocks.0.norm2.we

In [10]:
def iter_chunks(y: np.ndarray, sr: int, chunk_sec: float = CHUNK_SEC):
    """Yield (chunk_idx, start_sec, end_sec, waveform) for non-overlapping windows."""
    n = int(chunk_sec * sr)
    if n <= 0:
        raise ValueError("chunk length must be positive")
    if len(y) == 0:
        return
    chunk_idx = 0
    for start in range(0, len(y), n):
        segment = y[start : start + n]
        if len(segment) == 0:
            continue
        start_sec = start / sr
        end_sec = (start + len(segment)) / sr
        yield chunk_idx, start_sec, end_sec, segment.astype(np.float32, copy=False)
        chunk_idx += 1


def load_mono_48k(path: Path, sr: int = SR) -> np.ndarray:
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32, copy=False)


def embed_chunk(model, waveform: np.ndarray) -> np.ndarray:
    # laion_clap expects a list/batch of 1-D float arrays
    with torch.no_grad():
        emb = model.get_audio_embedding_from_data(x=[waveform], use_tensor=False)
    return np.asarray(emb[0], dtype=np.float32)

In [11]:
done_ids: set[str] = set()
embeddings: list[np.ndarray] = []
rows: list[dict] = []

if OUT_META.is_file() and OUT_EMB.is_file():
    prev_meta = pd.read_csv(OUT_META)
    prev_emb = np.load(OUT_EMB)
    if len(prev_meta) != len(prev_emb):
        raise ValueError(
            f"Resume mismatch: meta rows={len(prev_meta)} emb rows={len(prev_emb)}. "
            "Delete outputs and re-run."
        )
    done_ids = set(prev_meta["episode_id"].astype(str))
    embeddings = [prev_emb[i] for i in range(len(prev_emb))]
    rows = prev_meta.to_dict(orient="records")
    print(f"Resuming: {len(done_ids):,} episodes already embedded ({len(rows):,} chunks)")
else:
    print("Starting fresh (no existing outputs)")

errors: list[dict] = []

for path in tqdm(audio_paths, desc="CLAP 10s embed"):
    episode_id = path.stem
    if episode_id in done_ids:
        continue

    try:
        y = load_mono_48k(path)
        yam = yamnet.loc[episode_id] if episode_id in yamnet.index else None

        for chunk_idx, start_sec, end_sec, segment in iter_chunks(y, SR, CHUNK_SEC):
            vec = embed_chunk(clap_model, segment)
            embeddings.append(vec)

            row = {
                "episode_id": episode_id,
                "chunk_idx": chunk_idx,
                "start_sec": float(start_sec),
                "end_sec": float(end_sec),
                "duration_sec": float(end_sec - start_sec),
                "source_path": str(path.as_posix()),
            }
            if yam is not None:
                for col in YAMNET_COLS:
                    val = yam[col]
                    row[col] = None if pd.isna(val) else val
            else:
                for col in YAMNET_COLS:
                    row[col] = None
            rows.append(row)

        done_ids.add(episode_id)
    except Exception as e:
        errors.append({"episode_id": episode_id, "error": str(e)})
        continue

print(f"chunks embedded: {len(embeddings):,}")
print(f"episodes done: {len(done_ids):,}")
print(f"errors: {len(errors):,}")
if errors:
    pd.DataFrame(errors).head(10)

Starting fresh (no existing outputs)


CLAP 10s embed: 100%|██████████| 6374/6374 [32:53<00:00,  3.23it/s]  

chunks embedded: 35,312
episodes done: 6,374
errors: 0


In [12]:
if not embeddings:
    raise RuntimeError("No embeddings produced")

emb_arr = np.stack(embeddings, axis=0).astype(np.float32, copy=False)
meta_df = pd.DataFrame(rows)

if len(meta_df) != emb_arr.shape[0]:
    raise ValueError(f"Row mismatch: meta={len(meta_df)} emb={emb_arr.shape[0]}")

OUT_EMB.parent.mkdir(parents=True, exist_ok=True)
np.save(OUT_EMB, emb_arr)
meta_df.to_csv(OUT_META, index=False)

print(f"saved embeddings: {OUT_EMB.resolve()} shape={emb_arr.shape}")
print(f"saved metadata:   {OUT_META.resolve()} rows={len(meta_df):,}")

saved embeddings: C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data\clap_10s_embeddings.npy shape=(35312, 512)
saved metadata:   C:\Users\utkar\OneDrive\Desktop\Northwestern University\Studies\NLP\Gen AI\SaysUserContentRecommendation\data\clap_10s_metadata.csv rows=35,312


In [13]:
assert OUT_EMB.is_file() and OUT_META.is_file()
check_emb = np.load(OUT_EMB)
check_meta = pd.read_csv(OUT_META)
assert len(check_meta) == check_emb.shape[0]

print("embeddings:", check_emb.shape, check_emb.dtype)
print("metadata columns:", list(check_meta.columns))
print("episodes:", check_meta["episode_id"].nunique())
print("chunks with YAMNet rank-1:", check_meta["yamnet_rank_1_class"].notna().sum())
check_meta[["episode_id", "chunk_idx", "start_sec", "end_sec", "duration_sec", "yamnet_rank_1_class", "yamnet_rank_2_class"]].head(10)

embeddings: (35312, 512) float32
metadata columns: ['episode_id', 'chunk_idx', 'start_sec', 'end_sec', 'duration_sec', 'source_path', 'yamnet_rank_1_class', 'yamnet_rank_1_mean_score', 'yamnet_rank_1_max_score', 'yamnet_rank_2_class', 'yamnet_rank_2_mean_score', 'yamnet_rank_2_max_score', 'yamnet_rank_3_class', 'yamnet_rank_3_mean_score', 'yamnet_rank_3_max_score', 'yamnet_rank_4_class', 'yamnet_rank_4_mean_score', 'yamnet_rank_4_max_score', 'yamnet_rank_5_class', 'yamnet_rank_5_mean_score', 'yamnet_rank_5_max_score', 'yamnet_top5_json']
episodes: 6374
chunks with YAMNet rank-1: 35307


,episode_id,chunk_idx,start_sec,end_sec,duration_sec,yamnet_rank_1_class,yamnet_rank_2_class
0,00057e4c-6542-4bed-be04-a308002dda40,0,0.0,10.0,10.0,Music,Speech
1,00057e4c-6542-4bed-be04-a308002dda40,1,10.0,20.0,10.0,Music,Speech
2,00057e4c-6542-4bed-be04-a308002dda40,2,20.0,30.0,10.0,Music,Speech
3,00057e4c-6542-4bed-be04-a308002dda40,3,30.0,40.0,10.0,Music,Speech
4,00057e4c-6542-4bed-be04-a308002dda40,4,40.0,50.0,10.0,Music,Speech
5,00057e4c-6542-4bed-be04-a308002dda40,5,50.0,60.0,10.0,Music,Speech
6,00057e4c-6542-4bed-be04-a308002dda40,6,60.0,70.0,10.0,Music,Speech
7,00057e4c-6542-4bed-be04-a308002dda40,7,70.0,80.0,10.0,Music,Speech
8,00057e4c-6542-4bed-be04-a308002dda40,8,80.0,90.0,10.0,Music,Speech
9,00057e4c-6542-4bed-be04-a308002dda40,9,90.0,100.0,10.0,Music,Speech
